In [ ]:
import xarray as xr
import dask.dataframe as dd
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from data_sparsity.generate_data import GenerateData


def case_paths(case_slug):
    case_root = Path("./tutorial1") / case_slug
    return (
        str(case_root / "netCDF" / "ds.nc"),
        str(case_root / "parquet" / "ddf"),
        str(case_root / "parquet" / "tmp"),
    )


def format_bytes(num_bytes):
    units = ["B", "KiB", "MiB", "GiB"]
    size = float(num_bytes)
    for unit in units:
        if size < 1024 or unit == units[-1]:
            return f"{size:.2f} {unit}"
        size /= 1024


def parquet_disk_size(parquet_path):
    path = Path(parquet_path)
    if not path.exists():
        path = path.parent
    if path.is_file():
        return path.stat().st_size
    return sum(path_item.stat().st_size for path_item in path.rglob("*") if path_item.is_file())


def format_table_value(value):
    if isinstance(value, float):
        return f"{value:.3f}"
    return str(value)


def draw_table(ax, cell_text, col_labels, row_labels=None, title=None):
    ax.set_axis_off()
    table = ax.table(
        cellText=cell_text,
        colLabels=col_labels,
        rowLabels=row_labels,
        cellLoc="center",
        loc="center",
    )
    table.auto_set_font_size(False)
    table.set_fontsize(7)
    table.scale(1, 1.2)
    for (row, col), cell in table.get_celld().items():
        cell.set_edgecolor("dimgray")
        if row == 0:
            cell.set_facecolor("#e6e6e6")
            cell.set_text_props(color="dimgray", weight="bold")
        elif col == -1:
            cell.set_facecolor("#f5f5f5")
            cell.set_text_props(color="dimgray")
    if title is not None:
        ax.set_title(title, color="dimgray", pad=6)
    return table


def draw_storage_schema(ax, ds, df):
    occupied_sites = len(df)
    num_dims = len(ds.dims)
    num_vars = len(ds.data_vars)
    grid_shape = [int(ds.sizes[dim]) for dim in ds.dims]
    total_grid_points = int(np.prod(grid_shape))
    tabular_rows = occupied_sites
    tabular_values = occupied_sites * (num_dims + num_vars)
    array_coord_values = sum(grid_shape)
    array_values = total_grid_points * num_vars
    array_missing = total_grid_points - occupied_sites

    ax.set_axis_off()
    ax.text(
        0.5,
        0.98,
        "Storage schema",
        ha="center",
        va="top",
        color="dimgray",
        fontsize=12,
        weight="bold",
        transform=ax.transAxes,
    )
    ax.text(
        0.5,
        0.93,
        f"occupied sites: {occupied_sites} | total grid points: {total_grid_points}",
        ha="center",
        va="top",
        color="dimgray",
        fontsize=8,
        transform=ax.transAxes,
    )

    tab_ax = ax.inset_axes([0.02, 0.53, 0.96, 0.34])
    arr_ax = ax.inset_axes([0.02, 0.07, 0.96, 0.34])

    tabular_preview = df[[col for col in df.columns if col.startswith("x") or col == "record"]].head(occupied_sites)
    tabular_cell_text = [[format_table_value(value) for value in row] for row in tabular_preview.to_numpy()]
    draw_table(
        tab_ax,
        tabular_cell_text,
        list(tabular_preview.columns),
        title=(
            f"Tabular: {tabular_rows} rows | {tabular_values} stored values\n"
            f"{occupied_sites * num_dims} repeated coordinate values + {occupied_sites * num_vars} data values"
        ),
    )

    record = ds["record"].values
    array_cell_text = [["NaN" if np.isnan(value) else f"{value:.3f}" for value in row] for row in record]
    draw_table(
        arr_ax,
        array_cell_text,
        [format_table_value(value) for value in ds["x1"].values],
        row_labels=[format_table_value(value) for value in ds["x0"].values],
        title=(
            f"Array: {array_coord_values} coordinate values + {array_values} grid values\n"
            f"empty sites stored as NaN: {array_missing}"
        ),
    )

    return ax


def plot_grid_case(ds, df, title):
    x0 = ds["x0"].values
    x1 = ds["x1"].values
    support_x1, support_x0 = np.meshgrid(x1, x0)
    present_mask = np.isfinite(ds["record"].values)

    fig = plt.figure(figsize=(13, 5))
    outer = fig.add_gridspec(1, 2, width_ratios=[1.1, 1.0], wspace=0.15)
    ax = fig.add_subplot(outer[0, 0])
    ax.scatter(
        support_x1.ravel(),
        support_x0.ravel(),
        s=160,
        facecolors="none",
        edgecolors="dimgray",
        linewidths=1.2,
        zorder=2,
    )

    for xv in x1:
        ax.axvline(xv, color="dimgray", linestyle=":", linewidth=1, zorder=0)
    for yv in x0:
        ax.axhline(yv, color="dimgray", linestyle=":", linewidth=1, zorder=0)

    ax.scatter(
        support_x1.ravel()[present_mask.ravel()],
        support_x0.ravel()[present_mask.ravel()],
        marker="x",
        c="green",
        s=90,
        linewidths=2,
        zorder=3,
    )

    ax.set_xticks(x1)
    ax.set_yticks(x0)
    ax.set_xticklabels([f"{value:.3f}" for value in x1], color="dimgray")
    ax.set_yticklabels([f"{value:.3f}" for value in x0], color="dimgray")
    ax.set_xlabel("x1", color="dimgray")
    ax.set_ylabel("x0", color="dimgray")
    ax.set_title(title, color="dimgray")
    ax.tick_params(axis="both", colors="dimgray")
    ax.set_aspect("equal", adjustable="box")
    for spine in ax.spines.values():
        spine.set_color("dimgray")
    ax.grid(False)

    schema_ax = fig.add_subplot(outer[0, 1])
    draw_storage_schema(schema_ax, ds, df)
    plt.tight_layout()
    plt.show()


def run_case(case_slug, num_obs, sparsity, seed, title):
    ncpath, pqpath, pqpathtmp = case_paths(case_slug)
    gen = GenerateData(
        num_obs=num_obs,
        num_dims=2,
        ratio_dims=1,
        sparsity=sparsity,
        seed=seed,
    )
    gen.generate(
        netcdf_filepath=ncpath,
        parquet_filepath=pqpath,
        parquet_tmp=pqpathtmp,
    )
    ds = xr.open_dataset(ncpath).load()
    df = pd.read_parquet(os.path.dirname(pqpath))

    nc_disk_bytes = Path(ncpath).stat().st_size
    pq_disk_bytes = parquet_disk_size(pqpath)
    ds_memory_bytes = ds.nbytes
    df_memory_bytes = df.memory_usage(index=True, deep=True).sum()

    print(f"Loaded netCDF into xarray: {format_bytes(ds_memory_bytes)} in memory")
    print(f"Loaded parquet into pandas: {format_bytes(df_memory_bytes)} in memory")
    print(f"On-disk netCDF size: {format_bytes(nc_disk_bytes)}")
    print(f"On-disk parquet size: {format_bytes(pq_disk_bytes)}")
    plot_grid_case(ds, df, title)
    return ds, df

### List of cases - Single variable

2D because it's easier to understand and visualize

* Purely gridded data, maximum density (minimum sparsity)
* Purely irregular data
* Maximum sparsity on a grid
* Somehow sparse data on a grid

Each case is written to its own folder and rendered as a representative map.

### Purely gridded data, 3x3 grid, 9 points

In [ ]:
ds, df = run_case(
    "purely_gridded",
    num_obs=9,
    sparsity=1,
    seed=202607,
    title="Purely gridded data",
)

### Purely irregular data, 3 points

In [ ]:
ds, df = run_case(
    "purely_irregular",
    num_obs=3,
    sparsity=1 / 3,
    seed=202607,
    title="Purely irregular data",
)


### Maximum sparsity on a grid, 3 points

In [ ]:
ds, df = run_case(
    "maximum_sparsity_grid",
    num_obs=3,
    sparsity=1 / 3,
    seed=202607,
    title="Maximum sparsity on a grid",
)


### Somehow sparse data on a grid, 6 points

In [ ]:
ds, df = run_case(
    "somehow_sparse_grid",
    num_obs=6,
    sparsity=6 / 9,
    seed=202607,
    title="Somehow sparse data on a grid",
)
